## a) Basic setup
1. python packages
2. API keys

In [ ]:
!pip install -q biopython requests pandas dotenv

In [ ]:
import os

# Detect environment and load secrets
try:
    from google.colab import userdata

    # --- Google Colab ---
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    ENTREZ_EMAIL = userdata.get("ENTREZ_EMAIL")
    NCBI_API_KEY = userdata.get("NCBI_API_KEY")

    print("Running in Google Colab.")

except ImportError:
    # --- Local Jupyter ---
    from dotenv import load_dotenv

    load_dotenv()  # reads .env from the notebook's directory

    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
    ENTREZ_EMAIL = os.getenv("ENTREZ_EMAIL")
    NCBI_API_KEY = os.getenv("NCBI_API_KEY")

    print("Running locally.")

# Validate required secrets
assert GEMINI_API_KEY, (
    "GEMINI_API_KEY is not configured -- get a free key at https://aistudio.google.com/api-keys"
)

assert ENTREZ_EMAIL, (
    "ENTREZ_EMAIL is not configured."
)

if NCBI_API_KEY:
    print("NCBI_API_KEY found.")
else:
    print("No NCBI_API_KEY found -- using the slower NCBI rate limit.")

NCBI_SLEEP = 0.11 if NCBI_API_KEY else 0.4



In [ ]:
from Bio import Entrez

Entrez.email = ENTREZ_EMAIL
if NCBI_API_KEY:
    Entrez.api_key = NCBI_API_KEY

# NCBI rate limit: ~3 req/s without an API key, ~10 req/s with one.
NCBI_SLEEP = 0.11 if NCBI_API_KEY else 0.4


---
## Before you start

Picks up after [01_query_corpus.ipynb](01_query_corpus.ipynb). That notebook builds and
cleans the PMID corpus; here we triage a small sample of papers from it. Swap in your own
PMIDs via `sample_pmids` below if you want.

---
## b) The triage schema — edit prompts directly here

`TRIAGE_FLOWCHART` and `TRIAGE_PROMPTS` below define the same shape as the real pipeline's JSON
config: flowchart nodes have `type`/`data.prompt_name`/`transitions`, prompts are
`{name, prompt, output_format}`. They're defined as plain Python dicts in this notebook (not
downloaded), so editing a prompt's wording is just editing the dict below and re-running this cell
-- no file to download, re-publish, or re-verify.

In [ ]:
import json

# Edit prompt wording directly here -- this is the one and only place these questions are
# defined (no download/checksum step). Re-run this cell after any edit for it to take effect.
# target_section lists which paper section(s) (IAO ontology codes) this question should see when
# triaging full text, section-aware (section e) -- TITLE=IAO:0000305, ABSTRACT=IAO:0000315,
# METHODS=IAO:0000317, RESULTS=IAO:0000318, BODY=IAO:0000314 (a catch-all for untyped body text).
TRIAGE_FLOWCHART = {
    "startNode": "check_bacteria",
    "nodes": {
        "check_bacteria": {
            "type": "conditional_prompt_boolean",
            "data": {"prompt_name": "is_about_bacteria"},
            "transitions": {"true": "check_gene_genotype", "false": "terminal_triage"},
        },
        "check_gene_genotype": {
            "type": "conditional_prompt_boolean",
            "data": {"prompt_name": "has_gene_genotype"},
            "transitions": {"true": "check_amr_phenotype", "false": "check_amr_phenotype"},
        },
        "check_amr_phenotype": {
            "type": "conditional_prompt_boolean",
            "data": {"prompt_name": "has_amr_phenotype"},
            "transitions": {"true": "terminal_triage", "false": "terminal_triage"},
        },
        "terminal_triage": {"type": "terminal_triage", "data": {}, "transitions": {}},
    },
}

_PROMPTS_RAW = {
    "prompts": [
        {
            "name": "is_about_bacteria",
            "target_section": ["IAO:0000305", "IAO:0000315", "IAO:0000317", "IAO:0000314"],
            "prompt": "Does this paper study bacteria or bacterial strains?",
            "output_format": "boolean",
        },
        {
            "name": "has_gene_genotype",
            "target_section": ["IAO:0000305", "IAO:0000315", "IAO:0000317", "IAO:0000318", "IAO:0000314"],
            "prompt": ("Does this paper report a specific bacterial gene or genotype by name (e.g. a named "
                       "resistance gene, efflux pump gene, or genotype/lineage designation) in the bacterial "
                       "strain(s) under study, even if no specific sequence variant/mutation is described? "
                       "Answer NO if no gene or genotype is named for the study bacterium."),
            "output_format": "boolean",
        },
        {
            "name": "has_amr_phenotype",
            "target_section": ["IAO:0000305", "IAO:0000315", "IAO:0000317", "IAO:0000318", "IAO:0000314"],
            "prompt": ("Does this paper report a phenotype specifically related to antimicrobial/antibiotic "
                       "resistance or susceptibility for the bacterial strain(s) under study (e.g. MIC values, "
                       "resistant/susceptible/intermediate classification, resistance gene expression, inhibition "
                       "zone diameter)? Answer NO if the only phenotypes reported are unrelated to antimicrobial "
                       "resistance/susceptibility (e.g. growth rate, biofilm formation, virulence with no "
                       "antimicrobial/antibiotic resistance angle)."),
            "output_format": "boolean",
        },
    ],
}

TRIAGE_PROMPTS = {p["name"]: p for p in _PROMPTS_RAW["prompts"]}

print("Start node:", TRIAGE_FLOWCHART["startNode"])
print("Questions:", list(TRIAGE_PROMPTS.keys()))


## c) The Gemini call to triage

`call_llm` sends one prompt+paper-text pair to the Gemini API and parses the `{"answer": ..., "reasoning": ...}` JSON out of the response. `run_triage` walks `TRIAGE_FLOWCHART` node by node: at each `conditional_prompt_boolean` node call the LLM and branch on `true`/`false`, stop at `terminal_triage`.

In [ ]:
# SET the MODEL NAME you would like to use via the Gemini API, and its free-tier
# RPM (requests/minute) and RPD (requests/day) caps -- check current limits at
# https://ai.google.dev/gemini-api/docs/rate-limits if this model's caps change.
GEMINI_URL_TEMPLATE = "https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
GEMINI_MODEL = "gemini-3.5-flash-lite"
GEMINI_RPM = 15
GEMINI_RPD = 500

MIN_REQUEST_INTERVAL = 60.0 / GEMINI_RPM  # spaced out evenly across a minute
_last_call_end = 0.0
_daily_request_count = 0


In [ ]:
import re
import time

# Gemini can return 429 (per-minute/per-day quota) or a transient 5xx -- both are worth
# retrying with backoff rather than failing the whole run on one blip.
RETRYABLE_STATUS_CODES = {429, 500, 502, 503}


def _extract_json(text):
    """Model output is usually pure JSON, but strip stray markdown fences/prose defensively."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(match.group(0)) if match else {"answer": "no", "reasoning": f"unparsed: {text[:200]}"}


def _gemini_generate(user_content, model=None, max_retries=5):
    """Send one prompt to the Gemini API and return the raw text response -- the shared
    HTTP/retry/rate-limit plumbing behind call_llm (boolean triage). Callers own parsing.
    `model` defaults to GEMINI_MODEL (above) if not given."""
    global _last_call_end, _daily_request_count

    if _daily_request_count >= GEMINI_RPD:
        raise RuntimeError(
            f"Reached the configured daily cap (GEMINI_RPD={GEMINI_RPD}) for this session -- "
            "Gemini would reject further requests anyway. Wait until tomorrow, raise GEMINI_RPD "
            "if your actual quota is higher, or switch to the local model (section f)."
        )

    wait = MIN_REQUEST_INTERVAL - (time.time() - _last_call_end)
    if wait > 0:
        time.sleep(wait)

    url = GEMINI_URL_TEMPLATE.format(model=model or GEMINI_MODEL)

    for attempt in range(max_retries):
        try:
            resp = requests.post(
                url,
                params={"key": GEMINI_API_KEY},
                json={
                    "contents": [{"parts": [{"text": user_content}]}],
                    "generationConfig": {"temperature": 0},
                },
                timeout=120,
            )
        except requests.exceptions.RequestException as e:
            # A read timeout / connection drop never reaches resp.status_code -- without this,
            # one network blip raises straight out of here and (depending on the caller) can
            # abort an entire multi-paper loop instead of just this one call. Retry it exactly
            # like a transient 5xx.
            _last_call_end = time.time()
            backoff = 5.0 * (2 ** attempt)
            print(f"  network error ({e!r}), backing off {backoff:.0f}s "
                  f"(attempt {attempt + 1}/{max_retries})...")
            time.sleep(backoff)
            continue
        _last_call_end = time.time()
        _daily_request_count += 1

        if resp.status_code in RETRYABLE_STATUS_CODES:
            backoff = 5.0 * (2 ** attempt)
            print(f"  {resp.status_code} ({resp.text[:200]!r}), backing off {backoff:.0f}s "
                  f"(attempt {attempt + 1}/{max_retries})...")
            time.sleep(backoff)
            continue

        if not resp.ok:
            # Surface the actual reason (bad model id, invalid key, etc.) instead of a bare
            # HTTPError -- Gemini's error body usually says exactly what's wrong.
            print(f"  Gemini error {resp.status_code}: {resp.text[:500]}")
            resp.raise_for_status()

        return resp.json()["candidates"][0]["content"]["parts"][0]["text"]

    raise RuntimeError(
        "Gemini call failed after retries -- likely your free-tier rate limit (requests/minute "
        "or requests/day). Wait a bit and retry."
    )


def call_llm(question_prompt, paper_text, max_retries=5):
    user_content = (
        f"{question_prompt}\n\n"
        f"Paper text:\n{paper_text[:6000]}\n\n"
        'Respond ONLY with JSON in exactly this shape: {"answer": "yes" or "no", "reasoning": "one sentence"}'
    )
    content = _gemini_generate(user_content, max_retries=max_retries)
    parsed = _extract_json(content)
    answer = str(parsed.get("answer", "")).strip().lower() == "yes"
    return answer, parsed.get("reasoning", "")


def run_triage(paper_text, verbose=True):
    node_name = TRIAGE_FLOWCHART["startNode"]
    extraction_data = {}

    while True:
        node = TRIAGE_FLOWCHART["nodes"][node_name]
        if node["type"] == "terminal_triage":
            break

        prompt_name = node["data"]["prompt_name"]
        question_prompt = TRIAGE_PROMPTS[prompt_name]["prompt"]

        answer, reasoning = call_llm(question_prompt, paper_text)
        extraction_data[prompt_name] = answer
        if verbose:
            print(f"  [{prompt_name}] {answer} — {reasoning}")

        node_name = node["transitions"]["true" if answer else "false"]

    return {
        "is_bacteria": extraction_data.get("is_about_bacteria"),
        "has_gene_genotype": extraction_data.get("has_gene_genotype"),
        "has_amr_phenotype": extraction_data.get("has_amr_phenotype"),
    }


### Offline fallback: bundled sample papers

`sample_fulltext/` has pre-fetched text for the sample PMIDs, plus a `manifest.json` of
PMID -> PMCID. Used only if NCBI is unreachable — the fetch functions below always try
Entrez first. Doesn't cover PMIDs you add yourself.

In [ ]:
import json as _json

FALLBACK_DIR = "sample_fulltext"
with open(f"{FALLBACK_DIR}/manifest.json") as f:
    FALLBACK_MANIFEST = _json.load(f)

print(f"Loaded offline fallback manifest for {len(FALLBACK_MANIFEST)} PMIDs.")


## d) Get sample papers

The pipeline triages full-text BioC sections; for this self-contained demo we fetch title+abstract from PubMed for a handful of known AMR papers via `Bio.Entrez` — a reasonable simplification for a workshop exercise, not what the production pipeline does (it sees full text, and uses PMC's BioC XML/JSON export, not just the abstract).

In [ ]:
# A small curated set of real PMIDs — a mix of clear AMR papers and one that isn't AMR-focused,
# so triage has something to actually discriminate on.
#sample_pmids = ["12737743", "12800222", "12927083", "12059906"]
sample_pmids = ['12737743', '12800222', '12927083', '12059906','34515028', '34485958', '34463568', '34461978', '34414415'] # these are reported in AMR portal


def fetch_title_abstract(pmid):
    """Falls back to sample_fulltext/ (see above) if NCBI is unreachable."""
    try:
        handle = Entrez.efetch(db="pubmed", id=pmid, rettype="abstract", retmode="xml")
        records = Entrez.read(handle)
        handle.close()
        article = records["PubmedArticle"][0]["MedlineCitation"]["Article"]
        title = str(article.get("ArticleTitle", ""))
        abstract_parts = article.get("Abstract", {}).get("AbstractText", [])
        abstract = "\n".join(str(p) for p in abstract_parts)
        return f"{title}\n\n{abstract}"
    except Exception as e:
        local_path = f"{FALLBACK_DIR}/{pmid}_title_abstract.txt"
        if os.path.exists(local_path):
            print(f"  NCBI fetch failed ({e!r}) -- using bundled local copy for PMID {pmid}.")
            with open(local_path) as f:
                return f.read()
        raise


papers = {}
for pmid in sample_pmids:
    papers[pmid] = fetch_title_abstract(pmid)
    time.sleep(NCBI_SLEEP)

print(f"Fetched {len(papers)} papers")
print(papers[sample_pmids[0]][:400])

### Run triage on title/abstract across several papers and tabulate

**Watch the free-tier cap**: Gemini's free tier is rate-limited per-minute and per-day (check current limits at https://ai.google.dev/gemini-api/docs/rate-limits) -- this loop uses up to 4 requests per paper (one per triage question), so don't run it against more than a handful of papers in one round.

In [ ]:
import requests
import pandas as pd

rows = []
for pmid, text in papers.items():
    print(f"Triaging PMID {pmid}...")
    try:
        result = run_triage(text, verbose=False)
        rows.append({"pmid": pmid, **result})
    except requests.exceptions.RequestException as e:
        print(f"  [ERROR] Failed to triage PMID {pmid} due to network/API error: {e}")

gemini_abstract_df = pd.DataFrame(rows)
gemini_abstract_df

---
## e) Triage full-text of the papers, section-aware

So far triage has run on title+abstract only, as one blob of text per question.For many of these papers, you would see that the pipeline fails at the first step (`is_about_bacteria`). If you will read the abstract, you will realize that these papers are in fact are about bacteria and the triage did not work as expected.

To improve the triage, instead routes each triage question to the full text instead of only title+abstract. Tailor the prompts to specific the paper section(s) listed in its `target_section`
(IAO ontology codes — see `TRIAGE_PROMPTS[...]["target_section"]` loaded in section (b)), using the full text of the paper, not just the abstract.

This section cetches a paper's full text directly from PMC as JATS XML (`Entrez.efetch(db="pmc", ...)`)
and splits it into `{iao_id: text}` by section eeading — entirely self-contained, using only the NCBI
APIs this notebook already depends on. **This only works for papers in PMC's open-access full-text
subset** — not every PMID has one; a PMID with no PMC full-text link, or a PMC record that's
metadata-only, will come back with few or no sections below (title/abstract may still work via the
`is_about_bacteria`/`has_gene_genotype` fallback to whatever sections were found).

In [ ]:
# PMC's Open Access subset, pre-segmented into passages by NCBI's own text-mining pipeline
# (https://www.ncbi.nlm.nih.gov/research/bionlp/APIs/BioC-PMC/) -- infons["section_type"] is a
# clean enum (TITLE/ABSTRACT/INTRO/METHODS/RESULTS/DISCUSS/FIG/TABLE/...), so no hand-rolled
# heading-substring matching is needed the way JATS XML parsing used to require. Anything
# outside the core narrative sections (FIG/TABLE/REF/...) is bucketed into the generic
# IAO:0000314 "body" code -- an inline TABLE passage can still carry real MIC/susceptibility
# data worth exposing to the genes/phenotypes curation stages.
SECTION_TYPE_TO_IAO = {
    "TITLE": "IAO:0000305",
    "ABSTRACT": "IAO:0000315",
    "INTRO": "IAO:0000316",
    "METHODS": "IAO:0000317",
    "RESULTS": "IAO:0000318",
    "DISCUSS": "IAO:0000319",
}
BODY_IAO = "IAO:0000314"

BIOC_CACHE_DIR = "bioc_cache"


def pmid_to_pmcid(pmid):
    """Looks up the PMC ID linked to a PubMed PMID; None if there isn't one.
    Falls back to the local manifest if NCBI is unreachable."""
    try:
        handle = Entrez.elink(dbfrom="pubmed", db="pmc", id=pmid)
        result = Entrez.read(handle)
        handle.close()
        return f"PMC{result[0]['LinkSetDb'][0]['Link'][0]['Id']}"
    except (IndexError, KeyError):
        return None
    except Exception as e:
        entry = FALLBACK_MANIFEST.get(pmid)
        if entry:
            print(f"  NCBI elink failed ({e!r}) -- using bundled manifest for PMID {pmid}.")
            return entry.get("pmcid")
        return None


def _bioc_fulltext_json(pmcid):
    """Fetch a PMC article as BioC JSON and cache it to bioc_cache/ so later cells -- and
    other notebooks run from this same working directory -- don't refetch it. Falls back to
    the bundled sample_fulltext/ copy if NCBI is unreachable or the article isn't in PMC's
    Open Access BioC subset (narrower than what Entrez.efetch(db="pmc") alone would serve --
    some PMCIDs come back with title+abstract only, same limitation the old JATS-XML fetch
    had for these same papers)."""
    os.makedirs(BIOC_CACHE_DIR, exist_ok=True)
    cache_path = os.path.join(BIOC_CACHE_DIR, f"{pmcid}_fulltext.json")
    if os.path.exists(cache_path):
        return json.load(open(cache_path))

    url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode"
    try:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        data = json.loads(resp.text)
        # NCBI wraps the BioCCollection in a top-level list, even for a single document.
        if not (isinstance(data, list) and data and data[0].get("documents")):
            raise ValueError("empty/unexpected BioC response")
        json.dump(data, open(cache_path, "w"))
        return data
    except Exception as e:
        local_path = f"{FALLBACK_DIR}/{pmcid}_fulltext.json"
        if os.path.exists(local_path):
            print(f"  NCBI BioC fetch failed ({e!r}) -- using bundled local copy for {pmcid}.")
            return json.load(open(local_path))
        return None


def fetch_sections_from_pmc(pmcid):
    """Fetches a PMC article as BioC JSON (see _bioc_fulltext_json) and groups passage text
    by IAO section id via SECTION_TYPE_TO_IAO. Returns {} if the article isn't in PMC's Open
    Access BioC subset and no local fallback exists."""
    data = _bioc_fulltext_json(pmcid)
    if not data:
        return {}

    sections = {}
    for collection in data:
        for doc in collection.get("documents", []):
            for passage in doc.get("passages", []):
                text = passage.get("text", "")
                if not text:
                    continue
                section_type = passage.get("infons", {}).get("section_type", "")
                iao_id = SECTION_TYPE_TO_IAO.get(section_type, BODY_IAO)
                sections[iao_id] = (sections.get(iao_id, "") + "\n" + text).strip()

    return sections


def select_target_text(sections, target_section_ids, max_chars=6000):
    """Concatenate only the sections a prompt cares about; fall back to
    everything available if none of its target sections were found."""
    matched = [sections[i] for i in target_section_ids if i in sections]
    text = "\n\n".join(matched) if matched else "\n\n".join(sections.values())
    return text[:max_chars]


def run_triage_sectioned(sections, verbose=True):
    """Same flowchart walk as run_triage, but each question only sees the
    section(s) listed in its own target_section fnstead of one flat blob."""
    node_name = TRIAGE_FLOWCHART["startNode"]
    extraction_data = {}

    while True:
        node = TRIAGE_FLOWCHART["nodes"][node_name]
        if node["type"] == "terminal_triage":
            break

        prompt_name = node["data"]["prompt_name"]
        prompt_def = TRIAGE_PROMPTS[prompt_name]
        paper_text = select_target_text(sections, prompt_def.get("target_section", []))

        answer, reasoning = call_llm(prompt_def["prompt"], paper_text)
        extraction_data[prompt_name] = answer
        if verbose:
            print(f"  [{prompt_name}] {answer} — {reasoning}")

        node_name = node["transitions"]["true" if answer else "false"]

    return {
        "is_bacteria": extraction_data.get("is_about_bacteria"),
        "has_gene_genotype": extraction_data.get("has_gene_genotype"),
        "has_amr_phenotype": extraction_data.get("has_amr_phenotype"),
    }

### Try it

Pick one or all of your `sample_pmids`, look up its PMC ID, and fetch+triage the full text.

In [ ]:
rows = []
for pmid, text in papers.items():
    print(f"Triaging PMID {pmid}...")
    pmcid = pmid_to_pmcid(pmid)
    
    if pmcid is None:
        print(f"PMID {pmid} has no linked PMC full-text record -- try a different PMID.")
    else:
        print(f"PMID {pmid} -> {pmcid}")
        sections = fetch_sections_from_pmc(pmcid)
        print("Sections found:", list(sections.keys()))
    
        if sections:
            result = run_triage_sectioned(sections)
            print("\nResult:", result)
            rows.append({"pmid": pmid, **result})
        else:
            print("No usable sections came back (metadata-only or closed-access record)")
gemini_sectioned_df = pd.DataFrame(rows)
gemini_sectioned_df

---
## f) Advanced: host an open weight model on Colab or your personal laptop

Everything so far has called the Gemini API, which is rate-limited on the free tier and depends on
a third party staying up. This section fnstead downloads a small open-weight model and runs it
**locally** -- inside this Colab VM, or on your own laptop if you're running the notebook there (see
"Running it locally" in the course README).

The next cell detects what you're running on and picks a backend automatically, and sets the model
size once as `QWEN_MODEL_SIZE` -- that single setting is all you need to change; everything else
(the MLX repo, or the GGUF repo/filename) derives from it automatically:
- **Apple Silicon Mac** (M-series chip) -> `mlx-lm`, Apple's native framework -- fastest on that hardware.
- **Anything else** (Colab, Linux, Intel Mac) -> `llama.cpp` with a GGUF model -- works everywhere,
  CPU-only is fine.

`QWEN_MODEL_SIZE = "14B"` is the default -- comfortable on 24GB+ RAM (e.g. an M4 Pro laptop) or Colab.
Change it to `"8B"` or `"4B"` (smaller, faster, slightly less sharp) if 14B is too slow on your machine.

In [ ]:
import platform

IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"
LOCAL_BACKEND = "mlx" if IS_APPLE_SILICON else "llama_cpp"

# Set the model size ONCE here -- everything below (MLX repo, GGUF repo/filename) derives from
# this, so there's nothing else to change if you want a smaller/faster model.
QWEN_MODEL_SIZE = "14B"  # swap to "8B" or "4B" for a smaller/faster model

print(f"Detected platform: {platform.system()} {platform.machine()} -> using the {LOCAL_BACKEND!r} backend")
print(f"Model size: Qwen3-{QWEN_MODEL_SIZE}")


In [ ]:
if LOCAL_BACKEND == "mlx":
    !pip install -q mlx-lm
    MLX_MODEL_REPO = f"mlx-community/Qwen3-{QWEN_MODEL_SIZE}-4bit"
else:
    !pip install -q llama-cpp-python huggingface_hub
    from huggingface_hub import list_repo_files

    LOCAL_MODEL_REPO = f"Qwen/Qwen3-{QWEN_MODEL_SIZE}-GGUF"
    print(f"GGUF files available in {LOCAL_MODEL_REPO}:")
    for f in list_repo_files(LOCAL_MODEL_REPO):
        if f.endswith(".gguf"):
            print(" ", f)


In [ ]:
if LOCAL_BACKEND == "llama_cpp":
    from huggingface_hub import hf_hub_download

    # Q4_K_M is the usual speed/quality sweet spot -- change QWEN_MODEL_SIZE above, not this,
    # if you want a different model size. Only swap this filename directly if you want a
    # different quantization of the same size (check the file list printed above).
    LOCAL_MODEL_FILE = f"Qwen3-{QWEN_MODEL_SIZE}-Q4_K_M.gguf"

    # hf_hub_download already checks the local HF cache and skips re-downloading if this file
    # was fetched before -- no separate try_to_load_from_cache check needed.
    model_path = hf_hub_download(repo_id=LOCAL_MODEL_REPO, filename=LOCAL_MODEL_FILE)
    print("Model path:", model_path)
else:
    print("Using MLX -- no separate download step; the model downloads inside load() below.")


In [ ]:
if LOCAL_BACKEND == "mlx":
    from mlx_lm import load as mlx_load

    mlx_model, mlx_tokenizer = mlx_load(MLX_MODEL_REPO)
    print(f"Loaded {MLX_MODEL_REPO} via MLX.")
else:
    import os
    from llama_cpp import Llama

    llm_local = Llama(
        model_path=model_path,
        # n_ctx=0 tells llama.cpp to use the model's OWN trained context length (from the
        # GGUF's metadata) instead of guessing a fixed number -- for Qwen3 that's 32K+. This
        # is the full budget prompt + thinking + answer all share; no more tuning this value
        # by hand when ENABLE_LOCAL_THINKING (section f) or a long paper needs more room.
        n_ctx=0,
        n_threads=os.cpu_count(),
        verbose=False,
    )
    print("Loaded via llama.cpp.")


In [ ]:
import re as _re_local

ENABLE_LOCAL_THINKING = True  # set True to let the local model reason in a <think> block
                                # before answering, and KEEP that reasoning (see
                                # LOCAL_THINKING_TRACES below) instead of suppressing/discarding
                                # it. Thinking mode is slower and uses more tokens per call --
                                # bump max_tokens on the call you're using if the model's answer
                                # gets cut off before the closing JSON.

LOCAL_THINKING_TRACES = []  # one row per call that actually emitted a <think> block:
                             # {"label": ..., "prompt_name": ..., "thinking": ...}. "label" is
                             # whatever trace_label/pmid the caller passed in -- inspect this
                             # list directly, or build a DataFrame: pd.DataFrame(LOCAL_THINKING_TRACES)


def split_thinking(response):
    """Split a <think>...</think> block (Qwen3 and similar reasoning models) from the rest of
    the response. Returns (thinking_text_or_None, rest_of_response) -- thinking_text is None if
    no <think> block was found (e.g. ENABLE_LOCAL_THINKING is False and _local_generate's
    suppression worked, or this model doesn't emit one)."""
    match = _re_local.search(r"<think>(.*?)</think>", response, flags=_re_local.DOTALL)
    if match:
        rest = response[:match.start()] + response[match.end():]
        return match.group(1).strip(), rest.strip()
    # A still-open <think> with no closing tag means max_tokens cut generation off mid-thought --
    # treat everything after <think> as thinking, with no answer left to parse.
    match = _re_local.search(r"<think>(.*)$", response, flags=_re_local.DOTALL)
    if match:
        return match.group(1).strip(), ""
    return None, response.strip()


def strip_thinking(response):
    """Backward-compatible helper: just the non-thinking part of split_thinking."""
    return split_thinking(response)[1]


def _local_generate(user_content, max_tokens=200):
    """Backend-agnostic text generation -- hides the mlx-lm vs llama.cpp difference from
    call_llm_local, so run_triage_local/run_triage_local_sectioned don't need to care which
    backend section (f) picked. When ENABLE_LOCAL_THINKING is False (default), suppresses the
    <think> block at generation time (cheaper and more reliable than stripping it after the
    fact); when True, lets it through so callers can record it via LOCAL_THINKING_TRACES."""
    if LOCAL_BACKEND == "mlx":
        from mlx_lm import generate as mlx_generate

        messages = [{"role": "user", "content": user_content}]
        try:
            # enable_thinking is Qwen3's own chat-template switch for the <think> block.
            # Falls back to no kwarg if this tokenizer's template doesn't support it.
            prompt = mlx_tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, enable_thinking=ENABLE_LOCAL_THINKING
            )
        except TypeError:
            prompt = mlx_tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        # mlx-lm's generate()/sampler API has changed across versions -- if this errors with an
        # unexpected keyword, check `pip show mlx-lm` and the mlx-lm README for the current
        # signature (e.g. newer versions take a `sampler=` object instead of `temp=`).
        return mlx_generate(mlx_model, mlx_tokenizer, prompt=prompt, max_tokens=max_tokens, verbose=False)
    else:
        # /no_think is Qwen3's own in-text switch to skip the <think> block; omit it when
        # thinking mode is wanted.
        suffix = "" if ENABLE_LOCAL_THINKING else " /no_think"
        resp = llm_local.create_chat_completion(
            messages=[{"role": "user", "content": user_content + suffix}],
            temperature=0,
            max_tokens=max_tokens,
        )
        return resp["choices"][0]["message"]["content"]


def _record_thinking(thinking, label, prompt_name):
    if thinking:
        LOCAL_THINKING_TRACES.append({"label": label, "prompt_name": prompt_name, "thinking": thinking})


def call_llm_local(question_prompt, paper_text, max_tokens=8192, trace_label=None):
    # Generous flat budget -- n_ctx=0 above (llama.cpp) / the model's own default
    # (MLX) give plenty of context room, so this just needs to not be the bottleneck.
    """Same prompt contract as call_llm (section c), answered by the local model instead.
    trace_label (e.g. a pmid) tags any captured thinking trace in LOCAL_THINKING_TRACES."""
    user_content = (
        f"{question_prompt}\n\n"
        f"Paper text:\n{paper_text[:6000]}\n\n"
        'Respond ONLY with JSON in exactly this shape: {"answer": "yes" or "no", "reasoning": "one sentence"}'
    )
    content = _local_generate(user_content, max_tokens=max_tokens)
    thinking, content = split_thinking(content)
    _record_thinking(thinking, trace_label, question_prompt[:60])
    parsed = _extract_json(content)
    answer = str(parsed.get("answer", "")).strip().lower() == "yes"
    return answer, parsed.get("reasoning", "")


def run_triage_local(paper_text, verbose=True, pmid=None):
    """Same flowchart walk as run_triage (section c), using the local model via call_llm_local.
    pmid (if given) tags any captured thinking trace for this paper in LOCAL_THINKING_TRACES."""
    node_name = TRIAGE_FLOWCHART["startNode"]
    extraction_data = {}

    while True:
        node = TRIAGE_FLOWCHART["nodes"][node_name]
        if node["type"] == "terminal_triage":
            break

        prompt_name = node["data"]["prompt_name"]
        question_prompt = TRIAGE_PROMPTS[prompt_name]["prompt"]

        answer, reasoning = call_llm_local(question_prompt, paper_text, trace_label=pmid)
        extraction_data[prompt_name] = answer
        if verbose:
            print(f"  [{prompt_name}] {answer} — {reasoning}")

        node_name = node["transitions"]["true" if answer else "false"]

    return {
        "is_bacteria": extraction_data.get("is_about_bacteria"),
        "has_gene_genotype": extraction_data.get("has_gene_genotype"),
        "has_amr_phenotype": extraction_data.get("has_amr_phenotype"),
    }


def run_triage_local_sectioned(sections, verbose=True, pmid=None):
    """Section-aware version of run_triage_local -- mirrors run_triage_sectioned (section e),
    but answered by the local model via call_llm_local instead of Gemini. Each question
    only sees the section(s) listed in its own target_section, same as section e."""
    node_name = TRIAGE_FLOWCHART["startNode"]
    extraction_data = {}

    while True:
        node = TRIAGE_FLOWCHART["nodes"][node_name]
        if node["type"] == "terminal_triage":
            break

        prompt_name = node["data"]["prompt_name"]
        prompt_def = TRIAGE_PROMPTS[prompt_name]
        paper_text = select_target_text(sections, prompt_def.get("target_section", []))

        answer, reasoning = call_llm_local(prompt_def["prompt"], paper_text, trace_label=pmid)
        extraction_data[prompt_name] = answer
        if verbose:
            print(f"  [{prompt_name}] {answer} — {reasoning}")

        node_name = node["transitions"]["true" if answer else "false"]

    return {
        "is_bacteria": extraction_data.get("is_about_bacteria"),
        "has_gene_genotype": extraction_data.get("has_gene_genotype"),
        "has_amr_phenotype": extraction_data.get("has_amr_phenotype"),
    }


### Try it

Run every paper in `sample_pmids` through your local model. `USE_FULL_TEXT` below controls which
input it sees -- `True` (default) fetches full text and goes section-aware using the same
`fetch_sections_from_pmc`/`select_target_text` machinery from section (e); `False` uses title+abstract
only, same as `run_triage`/`run_triage_local` elsewhere. This only needs the local model loaded above;
no Gemini API call is involved.

In [ ]:
# Set if you would like to run triage on title+abstract only or full-text
USE_FULL_TEXT = True  # False -> title+abstract only (same input run_triage/run_triage_local use
                      # elsewhere); True -> full text, section-aware (section e's approach)

In [ ]:
def triage_local_for_pmid(pmid, use_full_text=USE_FULL_TEXT):
    """Triage one PMID with the local model only, either title+abstract or full-text
    section-aware depending on use_full_text."""
    if not use_full_text:
        if pmid not in papers:
            print(f"PMID {pmid}: not in `papers` (fetched in section d) -- skipping.")
            return {"pmid": pmid, "pmcid": None, "skipped_reason": "no abstract text available"}
        result = run_triage_local(papers[pmid], verbose=False, pmid=pmid)
        row = {"pmid": pmid, "pmcid": None, "skipped_reason": None}
        row.update(result)
        return row

    pmcid = pmid_to_pmcid(pmid)
    if pmcid is None:
        print(f"PMID {pmid}: no linked PMC full-text record -- skipping.")
        return {"pmid": pmid, "pmcid": None, "skipped_reason": "no PMC full-text record"}

    sections = fetch_sections_from_pmc(pmcid)
    if not sections:
        print(f"PMID {pmid} -> {pmcid}: no usable sections (metadata-only/closed-access) -- skipping.")
        return {"pmid": pmid, "pmcid": pmcid, "skipped_reason": "no usable sections"}

    print(f"PMID {pmid} -> {pmcid}, sections found: {list(sections.keys())}")
    result = run_triage_local_sectioned(sections, verbose=False, pmid=pmid)

    row = {"pmid": pmid, "pmcid": pmcid, "skipped_reason": None}
    row.update(result)
    return row


local_rows = [triage_local_for_pmid(pmid) for pmid in sample_pmids]
local_df = pd.DataFrame(local_rows)
local_df

# To try just one PMID instead of looping over all of sample_pmids:
# triage_local_for_pmid(sample_pmids[0])


### Optional: compare against Gemini

Compare the local-model results already sitting in `local_df` (the "Try it" cell above) against
the Gemini results already sitting in `gemini_abstract_df`/`gemini_sectioned_df` (sections c/e) --
no new API calls, local or remote. Picks whichever Gemini dataframe matches `USE_FULL_TEXT` so
both sides were triaged the same way (full text, section-aware, or title+abstract only).

In [ ]:
gemini_df = gemini_sectioned_df if USE_FULL_TEXT else gemini_abstract_df

compare_df = local_df.merge(
    gemini_df, on="pmid", suffixes=("_local", "_remote")
)
for col in ("is_bacteria", "has_gene_genotype", "has_amr_phenotype"):
    compare_df[f"{col}_agree"] = compare_df[f"{col}_local"] == compare_df[f"{col}_remote"]

compare_df


---
## g) Clean up: free the local model's memory

The local model stays loaded for the rest of the session otherwise. Set `FREE_LOCAL_MODEL`
to `True` below once nothing later still needs it, then run this cell.

In [ ]:
FREE_LOCAL_MODEL = True  # set True only once nothing later in this session still needs
                          # mlx_model/mlx_tokenizer/llm_local (local triage or curation) -- then
                          # re-run this cell to actually free them.

if not FREE_LOCAL_MODEL:
    print("FREE_LOCAL_MODEL is False -- skipping, local model (if loaded) stays resident. "
          "Set it to True above and re-run this cell when you're really done with it.")
else:
    import gc

    if "llm_local" in globals():
        del llm_local
        print("Freed llama.cpp model.")

    if "mlx_model" in globals():
        del mlx_model
        del mlx_tokenizer
        print("Freed MLX model.")

    gc.collect()

    if LOCAL_BACKEND == "mlx":
        try:
            import mlx.core as mx
            mx.clear_cache()
        except AttributeError:
            # Older mlx-core versions expose this under mx.metal instead.
            import mlx.core as mx
            mx.metal.clear_cache()

    print("Done.")
